# 03 — Readout decomposition

**The question.** A slide from the original project reported the bipartite HGNN beating GEARS by
17x, 27x and 76x on the three datasets. Beside it, GEARS' own training logs printed a genome-wide
validation MSE in the 0.001-0.006 range and a top-20-DE test MSE in the 0.07-0.25 range. The
slide's GEARS column sits in the top-20-DE range; the HGNN column sits in the genome-wide range.

That is an inference from magnitudes, not a verified fact — the code behind the slide is not
available. This notebook's job is to make it checkable: take **identical saved predictions** from
both models on the **same split**, and compute every readout on both.

Four things are shown, in order:

1. the full readout table for both models, mean over 5 seeds with 95% CIs;
2. the **ratio panel** — `top-20-DE MSE / genome-wide MSE`, computed for each model *separately*;
3. the **mismatched comparison**, drawn beside the correct one and labelled as the error it is;
4. Pearson on absolute expression vs on the delta vs on the DE delta, with the do-nothing baseline.

**Axis rule for this whole repository:** no bar axis starts anywhere but zero, and correlation panels
are pinned to [0, 1]. If an effect is only visible zoomed in, the caption says so.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "src"))

from pertreadout import evaluate, figures
from pertreadout.train import load_predictions

RUNS = REPO / "runs"
TABLES = REPO / "results" / "tables"

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

In [ ]:
per_run = evaluate.per_run_table(RUNS)
agg = evaluate.aggregate(per_run)
print(f"{per_run['seed'].nunique()} seeds x {per_run['dataset'].nunique()} datasets "
      f"x {per_run['model'].nunique()} model variants = {len(per_run)} runs")
per_run.head()

## 1. Every readout, both models, side by side

One call to `readouts.readout_table` produced every column below, from the same `predictions.npz`.
Nothing in this repository computes a metric anywhere else.

In [ ]:
wide = (agg[agg["model"].isin(["gears", "hgnn"])]
        .pivot_table(index=["dataset", "readout"], columns="model",
                     values=["mean", "ci_lo", "ci_hi"]))
wide.round(5)

### The like-for-like table

The same metric on both sides of every comparison. `overlapping_ci` is the only honest way to read a
gap this small: if the two intervals overlap, the ordering is not established by these runs.

In [ ]:
for readout in ["mse_genome_wide", "mse_top20_de", "pearson_delta"]:
    print(f"\n### {readout}")
    print(evaluate.like_for_like(agg, readout).to_string(index=False))

In [ ]:
fig = plt.imread(str(figures.fig_like_for_like(agg, REPO / "results" / "figures")))
plt.figure(figsize=(13, 4.5)); plt.imshow(fig); plt.axis("off"); plt.show()

## 2. The ratio panel

For **each model separately**, `top-20-DE MSE / genome-wide MSE`. Both bars come from that model's
own predictions, so nothing here is a comparison between models — it is a comparison between two
readouts.

The top-20 DE genes are *selected* for having the largest response to the perturbation. Their
residuals are correspondingly large, while the genome-wide average is dominated by the thousands of
genes that barely move. The ratio measures that selection, not the fit.

If the bars are similar heights across models, the ratio is a property of the metric pair and the
data — which means any "improvement factor" of that size can be manufactured from two identical
models by scoring one on DE genes and the other genome-wide.

In [ ]:
ratio = evaluate.ratio_band(per_run)
print(ratio.to_string(index=False))
band = ratio[ratio["model"].isin(["gears", "hgnn"])]["mean"]
print(f"\nmeasured ratio band across datasets and models: "
      f"{band.min():.0f}x - {band.max():.0f}x")

In [ ]:
fig = plt.imread(str(figures.fig_ratio_panel(ratio, REPO / "results" / "figures")))
plt.figure(figsize=(8, 5)); plt.imshow(fig); plt.axis("off"); plt.show()

## 3. The mismatched comparison, made explicit

**Left panel: this is the error.** GEARS' top-20-DE MSE against the HGNN's genome-wide MSE — two
different metrics, drawn as if they were one. **Right panel: the same predictions, same metric on
both sides.**

*The left panel is what you get if the two bars are different metrics.*

The table below names both metrics on both sides so the reader can see exactly which pairing
produces which number. This is the one place in the repository where a large ratio appears, and it
appears as an illustration of what metric pairing produces — not as a result.

In [ ]:
mismatch = evaluate.mismatched_comparison(agg)
print(mismatch.to_string(index=False))

In [ ]:
fig = plt.imread(str(figures.fig_mismatch(agg, REPO / "results" / "figures")))
plt.figure(figsize=(12, 5)); plt.imshow(fig); plt.axis("off"); plt.show()

## 4. Pearson: absolute, delta, DE delta

Pearson on **absolute expression** is close to free. The dashed line on the left panel is the score
of predicting the unperturbed control mean profile for every condition — a model with no
perturbation input at all. Any correlation reported on absolute expression has to be read against
that line, not against zero.

Pearson on the **delta** removes the shared baseline and asks whether the predicted *change* tracks
the true change. See Ahlmann-Eltze, Huber & Anders (Nature Methods 2025), who make the same point
about linear baselines; the "Pearson DE Delta" framing restricts it further to the genes that
actually moved.

All three panels use the same axis, [0, 1], so the collapse from the first to the second is visible
at its true size.

In [ ]:
ctrl_line = agg[agg["readout"] == "pearson_absolute_ctrl_baseline"][["dataset", "mean"]]
print("control-mean-profile Pearson (absolute expression), the score to beat:")
print(ctrl_line.groupby("dataset")["mean"].mean().round(4).to_string())

print("\nmodels:")
print(agg[agg["readout"].isin(["pearson_absolute", "pearson_delta", "pearson_delta_de"])]
      .pivot_table(index=["dataset", "model"], columns="readout", values="mean").round(4).to_string())

In [ ]:
fig = plt.imread(str(figures.fig_pearson_family(agg, REPO / "results" / "figures")))
plt.figure(figsize=(13, 4.5)); plt.imshow(fig); plt.axis("off"); plt.show()

## 5. The identity-graph ablation

Both architectures retrained with the GO and co-expression graphs replaced by identity graphs —
self-loops only, no prior information. Everything else is held fixed, and the same five seeds are
used, so the gap is attributable to the priors.

The source project's informal finding was that the GO prior contributed less than the architecture
assumed. Here it is a supported configuration with CIs, so the claim can be checked.

In [ ]:
abl = agg[(agg["readout"] == "mse_genome_wide")]
print(abl.pivot_table(index="dataset", columns="model", values="mean").round(5).to_string())

for base in ["gears", "hgnn"]:
    sub = abl[abl["model"].isin([base, base + "_noprior"])].pivot_table(
        index="dataset", columns="model", values="mean")
    if sub.shape[1] == 2:
        delta = 100 * (sub[base + "_noprior"] - sub[base]) / sub[base]
        print(f"\n{base}: genome-wide MSE change from removing the priors (%, positive = priors help)")
        print(delta.round(1).to_string())

In [ ]:
path = figures.fig_prior_ablation(agg, REPO / "results" / "figures")
if path is not None:
    plt.figure(figsize=(9, 5)); plt.imshow(plt.imread(str(path))); plt.axis("off"); plt.show()

## What this notebook establishes

Whatever the numbers above say, they say it about **the same predictions scored two ways**. That is
the contribution: the comparison is checkable, and it is checkable by anyone with the public data
and this repository.

See `results/tables/` for the CSVs behind every number here, and `04_direction_vs_effect_size.ipynb`
for the direction-accuracy question.